# 00 — Data audit (Stage A)

Discover mounted competition paths, confirm target schema from `sample_submission.csv`, count studies/series/labels, and write a deterministic fingerprint.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.metadata import audit_dataset, resolve_paths, save_audit, target_columns_from_sample, load_csv_optional
from src.utils import load_config

cfg = load_config(ROOT / "configs" / "submission_001.yaml")
paths = resolve_paths(explicit_root=cfg.get("paths", {}).get("competition_root"))
print("Competition root:", paths.root)
print(json.dumps(paths.to_dict(), indent=2))

In [ ]:
sample = load_csv_optional(paths.sample_submission)
assert sample is not None, "sample_submission.csv is required"
targets = target_columns_from_sample(sample)
print("Targets (authoritative order):", targets)
display(sample.head())

In [ ]:
audit = audit_dataset(paths, targets)
out = ROOT / "artifacts" / "audit.json"
save_audit(audit, out)
print("Fingerprint:", audit["fingerprint"]["fingerprint"])
print("Labeled studies:", audit["labeled_studies"])
print("Group column:", audit["group_column"])
print("TODOs:", audit["todos"])
print("Prevalence:")
print(json.dumps(audit["prevalence"], indent=2))